# Waze Churn Prediction — End-to-End Analytics and Machine Learning Project

This notebook consolidates the full Waze churn project completed across five courses of the Google Advanced Data Analytics Certificate.

It covers:
- Course 1: data inspection and early behavioral insights
- Course 2: exploratory data analysis and visualization
- Course 3: hypothesis testing
- Course 4: logistic regression baseline
- Course 5: Random Forest and XGBoost modeling

**Business objective:** predict monthly user churn and identify the behavioral patterns most associated with retention and churn.

**Dataset:** `waze_dataset.csv`

## Executive overview

Waze leadership wants to understand which users are most likely to churn, why they churn, and whether behavioral data can support a useful prediction model.

Across this project:
- the labeled modeling set contained **14,299 users**
- the target was imbalanced: **82.3% retained** vs **17.7% churned**
- behavioral engagement variables were more useful than device-related variables
- the best final model was **XGBoost**
- the final model improved on the logistic regression baseline, but still missed many churners, so it should not be deployed as a standalone production intervention tool in its current form

## PACE — Plan

### Problem statement
Predict monthly user churn and identify the main behavioral drivers behind churn.

### Goal
Build an interpretable baseline and a stronger tree-based model, then compare results and translate findings into business recommendations.

### Success criteria
- clean labeled dataset for modeling
- clear EDA findings linked to churn
- hypothesis test for device-related behavior differences
- logistic regression baseline
- Random Forest and XGBoost models
- stakeholder-ready recommendations

## Ethical considerations

1. We are being asked to build and evaluate machine learning models that predict whether a Waze user will churn or be retained, and to identify the variables that are most useful for that prediction.

2. The main ethical consideration is that model errors could lead to unfair or inefficient targeting of users. A false negative means a user who is likely to churn is missed, so Waze loses an opportunity to intervene and retain that user. A false positive means a user is incorrectly flagged as likely to churn, which could lead to unnecessary retention efforts or incentives being offered to users who would have stayed anyway.

3. The likely effect of a false negative is lost business value, because Waze may fail to retain users who are at risk of leaving. The likely effect of a false positive is relatively minor, because the company may spend extra effort or resources on a user who was not actually going to churn.

4. Yes, I would proceed with the request. The benefits of identifying at-risk users and improving retention outweigh the potential downsides, especially because the cost of false positives is low. However, the model should be monitored carefully and used as a decision-support tool rather than the only basis for action.

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier, plot_importance

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

In [ ]:
# Load dataset
df0 = pd.read_csv('waze_dataset.csv')

# Inspect the first rows
df0.head()

In [ ]:
# Inspect structure and summary statistics
print(df0.info())
display(df0.describe(include='all'))

## Course 1 — Data inspection and early insights

In [ ]:
# Missing values
missing_summary = df0.isna().sum().sort_values(ascending=False)
missing_summary

In [ ]:
# Compare rows with missing label vs. non-missing label
missing_label_stats = df0[df0['label'].isna()].describe(numeric_only=True)
non_missing_label_stats = df0[df0['label'].notna()].describe(numeric_only=True)

missing_label_stats.loc[['mean', '50%']]

In [ ]:
non_missing_label_stats.loc[['mean', '50%']]

In [ ]:
# Device distribution among missing labels
missing_device_counts = df0[df0['label'].isna()]['device'].value_counts(dropna=False)
missing_device_props = df0[df0['label'].isna()]['device'].value_counts(normalize=True, dropna=False)

display(missing_device_counts)
display(missing_device_props)

In [ ]:
# Class balance on labeled data
labeled_df = df0.dropna(subset=['label']).copy()

label_counts = labeled_df['label'].value_counts()
label_props = labeled_df['label'].value_counts(normalize=True)

display(label_counts)
display(label_props)

In [ ]:
# Early behavioral ratios used in initial inspection
course1_df = labeled_df.copy()

course1_df['km_per_drive'] = course1_df['driven_km_drives'] / course1_df['drives']
course1_df['km_per_driving_day'] = course1_df['driven_km_drives'] / course1_df['driving_days']
course1_df['drives_per_driving_day'] = course1_df['drives'] / course1_df['driving_days']

course1_df.replace([np.inf, -np.inf], 0, inplace=True)

course1_medians = course1_df.groupby('label')[
    ['driven_km_drives', 'duration_minutes_drives', 'activity_days',
     'km_per_drive', 'km_per_driving_day', 'drives_per_driving_day']
].median()

course1_medians

### Course 1 key findings
- The dataset contained **14,999 rows** and **13 columns**.
- Only the target variable `label` had missing values (**700 rows**).
- Missing labels did not show strong evidence of systematic bias.
- Churned users tended to drive **farther** and **longer**, but use the app on **fewer days**.
- Early ratio features suggested that **intensity per active day** may matter more than raw usage volume.

## Course 2 — Exploratory Data Analysis and visualization

In [ ]:
# Create a working copy for EDA
df_eda = df0.copy()

# Example engineered variable used in EDA
df_eda['km_per_driving_day'] = df_eda['driven_km_drives'] / df_eda['driving_days']
df_eda['km_per_driving_day'] = df_eda['km_per_driving_day'].replace([np.inf, -np.inf], 0)

# Labeled subset for churn analysis
eda_labeled = df_eda.dropna(subset=['label']).copy()

# Device and label proportions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

eda_labeled['device'].value_counts().plot(
    kind='pie', autopct='%.1f%%', ax=axes[0], ylabel='', title='Device Mix'
)
eda_labeled['label'].value_counts().plot(
    kind='pie', autopct='%.1f%%', ax=axes[1], ylabel='', title='Label Mix'
)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution checks for the main behavioral variables
num_cols = [
    'sessions', 'drives', 'total_sessions', 'n_days_after_onboarding',
    'driven_km_drives', 'duration_minutes_drives',
    'activity_days', 'driving_days'
]

fig, axes = plt.subplots(4, 2, figsize=(12, 16))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    sns.histplot(eda_labeled[col], bins=30, kde=False, ax=ax)
    ax.set_title(f'Distribution of {col}')

plt.tight_layout()
plt.show()

In [ ]:
# Validity checks
validity_summary = {
    'max_activity_days': eda_labeled['activity_days'].max(),
    'max_driving_days': eda_labeled['driving_days'].max(),
    'all_driving_days_lte_activity_days': bool((eda_labeled['driving_days'] <= eda_labeled['activity_days']).all())
}
validity_summary

In [ ]:
# Churn rate by driving_days
churn_rate_by_driving_days = (
    eda_labeled.assign(churn_flag=np.where(eda_labeled['label'] == 'churned', 1, 0))
    .groupby('driving_days', as_index=False)['churn_flag']
    .mean()
)

plt.figure(figsize=(10, 5))
sns.lineplot(data=churn_rate_by_driving_days, x='driving_days', y='churn_flag')
plt.title('Churn Rate by Driving Days')
plt.ylabel('Churn Rate')
plt.show()

In [ ]:
# Churn rate by binned km_per_driving_day
temp = eda_labeled.assign(
    churn_flag=np.where(eda_labeled['label'] == 'churned', 1, 0)
).copy()

temp = temp[temp['km_per_driving_day'] <= 1200]
temp['km_day_bin'] = pd.cut(temp['km_per_driving_day'], bins=20)

churn_rate_km_day = temp.groupby('km_day_bin', observed=False)['churn_flag'].mean().reset_index()

plt.figure(figsize=(14, 5))
sns.barplot(data=churn_rate_km_day, x='km_day_bin', y='churn_flag', color='steelblue')
plt.xticks(rotation=90)
plt.title('Churn Rate by km_per_driving_day (binned)')
plt.ylabel('Churn Rate')
plt.xlabel('km_per_driving_day bin')
plt.tight_layout()
plt.show()

### Course 2 key findings
- Most numeric variables were **right-skewed** and contained strong outliers.
- `activity_days` could reach **31**, while `driving_days` reached **30**.
- The rule `driving_days ≤ activity_days` held across the dataset.
- Churn tended to **increase** as `km_per_driving_day` increased.
- Churn tended to **decrease** as engagement frequency increased.
- Device type did not show meaningful churn differences in practical terms.

## Course 3 — Hypothesis testing
**Business question:** Do iPhone and Android users differ significantly in mean number of drives?

In [ ]:
# Welch's two-sample t-test for mean drives by device
iphone_drives = labeled_df.loc[labeled_df['device'] == 'iPhone', 'drives']
android_drives = labeled_df.loc[labeled_df['device'] == 'Android', 'drives']

t_stat, p_value = ttest_ind(iphone_drives, android_drives, equal_var=False)

hypothesis_results = pd.DataFrame({
    'group': ['iPhone', 'Android'],
    'mean_drives': [iphone_drives.mean(), android_drives.mean()]
})

display(hypothesis_results)
print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {p_value:.5f}')

### Course 3 conclusion
Since the p-value was greater than 0.05, we failed to reject the null hypothesis.

**Interpretation:** there was no statistically significant evidence that mean drives differed between iPhone and Android users. Device type was not a strong driver of behavioral differences in ride frequency.

## Course 4 — Logistic regression baseline

In [ ]:
# Prepare data for logistic regression
df_lr = df0.copy()

df_lr['km_per_driving_day'] = df_lr['driven_km_drives'] / df_lr['driving_days']
df_lr['km_per_driving_day'] = df_lr['km_per_driving_day'].replace([np.inf, -np.inf], 0)
df_lr['professional_driver'] = np.where(
    (df_lr['drives'] >= 60) & (df_lr['driving_days'] >= 15), 1, 0
)

df_lr = df_lr.dropna(subset=['label']).copy()

df_lr['device2'] = np.where(df_lr['device'] == 'iPhone', 1, 0)
df_lr['label2'] = np.where(df_lr['label'] == 'churned', 1, 0)

X_lr = df_lr.drop(columns=['ID', 'label', 'label2', 'device', 'sessions', 'driving_days'])
y_lr = df_lr['label2']

X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(
    X_lr, y_lr, test_size=0.25, stratify=y_lr, random_state=42
)

lr_model = LogisticRegression(penalty=None, max_iter=1000)
lr_model.fit(X_train_lr, y_train_lr)

lr_preds = lr_model.predict(X_test_lr)

lr_metrics = pd.DataFrame({
    'metric': ['accuracy', 'precision', 'recall', 'f1'],
    'value': [
        accuracy_score(y_test_lr, lr_preds),
        precision_score(y_test_lr, lr_preds),
        recall_score(y_test_lr, lr_preds),
        f1_score(y_test_lr, lr_preds),
    ]
})
lr_metrics

In [ ]:
# Logistic regression coefficients
coef_df = pd.DataFrame({
    'feature': X_train_lr.columns,
    'coefficient': lr_model.coef_[0]
}).sort_values('coefficient')

coef_df.tail(15)

### Course 4 documented results
The completed baseline logistic regression produced the following test performance:

- Accuracy: **0.8238**
- Precision: **0.518**
- Recall: **0.091**
- F1: **0.160**

Key interpretation:
- `activity_days` was the strongest predictor and had a **negative** relationship with churn probability.
- Longer tenure and professional-style usage were also associated with lower churn.
- Accuracy looked acceptable, but recall was too low for practical churn targeting.

## Course 5 — Random Forest and XGBoost

### PACE — Analyze / Construct
For the final machine learning stage, additional feature engineering was applied and the dataset was split into training, validation, and test sets using a **60/20/20** workflow.

In [ ]:
# Build final modeling dataframe
df = df0.copy()

df['km_per_driving_day'] = df['driven_km_drives'] / df['driving_days']
df['km_per_driving_day'] = df['km_per_driving_day'].replace([np.inf, -np.inf], 0)

df['percent_sessions_in_last_month'] = df['sessions'] / df['total_sessions']

df['professional_driver'] = np.where(
    (df['drives'] >= 60) & (df['driving_days'] >= 15), 1, 0
)

df['total_sessions_per_day'] = df['total_sessions'] / df['n_days_after_onboarding']

df['km_per_hour'] = df['driven_km_drives'] / (df['duration_minutes_drives'] / 60)

df['km_per_drive'] = df['driven_km_drives'] / df['drives']
df['km_per_drive'] = df['km_per_drive'].replace([np.inf, -np.inf], 0)

df['percent_of_sessions_to_favorite'] = (
    (df['total_navigations_fav1'] + df['total_navigations_fav2']) / df['total_sessions']
)

df = df.dropna(subset=['label']).copy()

df['device2'] = np.where(df['device'] == 'iPhone', 1, 0)
df['label2'] = np.where(df['label'] == 'churned', 1, 0)

df = df.drop(columns=['ID'])

X = df.drop(columns=['label', 'label2', 'device'])
y = df['label2']

X_train_interim, X_test, y_train_interim, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_interim, y_train_interim, test_size=0.25, stratify=y_train_interim, random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)
print(y_train.shape, y_val.shape, y_test.shape)

In [ ]:
# Helper functions
def make_results(model_name: str, model_object, metric: str):
    metric_dict = {
        'precision': 'mean_test_precision',
        'recall': 'mean_test_recall',
        'f1': 'mean_test_f1',
        'accuracy': 'mean_test_accuracy'
    }
    cv_results = pd.DataFrame(model_object.cv_results_)
    best_row = cv_results.iloc[cv_results[metric_dict[metric]].idxmax(), :]
    table = pd.DataFrame({
        'model': [model_name],
        'precision': [best_row.mean_test_precision],
        'recall': [best_row.mean_test_recall],
        'F1': [best_row.mean_test_f1],
        'accuracy': [best_row.mean_test_accuracy]
    })
    return table

def get_test_scores(model_name: str, preds, y_true):
    return pd.DataFrame({
        'model': [model_name],
        'precision': [precision_score(y_true, preds)],
        'recall': [recall_score(y_true, preds)],
        'F1': [f1_score(y_true, preds)],
        'accuracy': [accuracy_score(y_true, preds)]
    })

In [ ]:
# Random Forest
rf = RandomForestClassifier(random_state=42)

rf_params = {
    'max_depth': [5, 10, None],
    'max_features': ['sqrt'],
    'max_samples': [0.7, 1.0],
    'min_samples_leaf': [1, 2],
    'min_samples_split': [2, 5],
    'n_estimators': [100, 200]
}

scoring = ['precision', 'recall', 'f1', 'accuracy']

rf_cv = GridSearchCV(
    estimator=rf,
    param_grid=rf_params,
    scoring=scoring,
    cv=4,
    refit='recall'
)

rf_cv.fit(X_train, y_train)
rf_results = make_results('Random Forest CV', rf_cv, 'recall')
rf_results

In [ ]:
# Random Forest validation performance
rf_val_preds = rf_cv.best_estimator_.predict(X_val)
rf_val_results = get_test_scores('Random Forest Val', rf_val_preds, y_val)
rf_val_results

In [ ]:
# XGBoost
xgb = XGBClassifier(
    objective='binary:logistic',
    random_state=42,
    eval_metric='logloss'
)

xgb_params = {
    'max_depth': [4, 6],
    'min_child_weight': [1, 3],
    'learning_rate': [0.1],
    'n_estimators': [100, 200]
}

xgb_cv = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_params,
    scoring=scoring,
    cv=4,
    refit='recall'
)

xgb_cv.fit(X_train, y_train)
xgb_results = make_results('XGBoost CV', xgb_cv, 'recall')
xgb_results

In [ ]:
# XGBoost validation and test performance
xgb_val_preds = xgb_cv.best_estimator_.predict(X_val)
xgb_val_results = get_test_scores('XGBoost Val', xgb_val_preds, y_val)

xgb_test_preds = xgb_cv.best_estimator_.predict(X_test)
xgb_test_results = get_test_scores('XGBoost Test', xgb_test_preds, y_test)

results = pd.concat([rf_results, rf_val_results, xgb_results, xgb_val_results, xgb_test_results], axis=0)
results

### Documented Course 5 results from the completed project

| Model | Precision | Recall | F1 | Accuracy |
|---|---:|---:|---:|---:|
| Random Forest CV | 0.473981 | 0.107081 | 0.174684 | 0.820491 |
| Random Forest Val | 0.468750 | 0.118343 | 0.188976 | 0.819930 |
| XGBoost Val | 0.380682 | 0.132150 | 0.196193 | 0.808042 |
| XGBoost Test | 0.448864 | 0.155819 | 0.231332 | 0.816434 |

**Champion model:** XGBoost, selected on recall.

In [ ]:
# Confusion matrix for the champion model on the test set
cm = confusion_matrix(y_test, xgb_test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('XGBoost Confusion Matrix — Test Set')
plt.show()

cm

### Confusion matrix interpretation
The documented test-set confusion matrix was:

- True negatives: **2256**
- False positives: **97**
- False negatives: **428**
- True positives: **79**

The model predicted about three times as many false negatives as false positives, which means it still missed most actual churners.

In [ ]:
# Feature importance
plot_importance(xgb_cv.best_estimator_, max_num_features=10)
plt.title('XGBoost Feature Importance')
plt.show()

### Top feature importance insights
The most important features in the final XGBoost model were:

1. `km_per_hour`
2. `percent_sessions_in_last_month`
3. `n_days_after_onboarding`
4. `total_navigations_fav1`
5. `percent_of_sessions_to_favorite`
6. `total_sessions_per_day`
7. `km_per_driving_day`
8. `total_sessions`
9. `duration_minutes_drives`
10. `km_per_drive`

A major takeaway is that **engineered features accounted for many of the most important predictors**.

## Final conclusions and recommendations

### Would you recommend using this model for churn prediction?
I would not recommend using this model as a production churn prediction tool in its current form. Although the XGBoost model performed better than logistic regression and Random Forest on recall, it still identified only a small share of actual churners. On the test set, recall was about 0.156, which means the model missed most users who churned. It can still be useful as a directional tool for understanding churn drivers and guiding future modeling work.

### What tradeoff was made by splitting the data into training, validation, and test sets?
The tradeoff is that each partition contains fewer observations, so the model has less data to learn from. However, this approach produces a more reliable model selection process because the validation set helps compare candidate models and the test set provides a more honest estimate of final performance on unseen data.

### What is the benefit of using logistic regression over tree-based ensembles?
Logistic regression is more interpretable. Its coefficients make it easier to explain how each variable is associated with churn, which is useful when communicating with stakeholders and when transparency matters.

### What is the benefit of using tree-based ensembles over logistic regression?
Ensemble tree-based models such as Random Forest and XGBoost are better at capturing nonlinear relationships and feature interactions. They often provide stronger predictive performance than logistic regression when the underlying patterns are more complex.

### What could improve the model?
- threshold tuning
- class weighting
- resampling strategies
- broader hyperparameter search
- additional feature engineering
- richer behavioral and product-related variables

### What additional features would be most useful?
- app crash and performance data
- push notification exposure
- month-over-month usage trends
- route complexity
- geographic context
- traffic conditions
- support interactions
- in-app satisfaction or frustration indicators

## Final takeaway

This project demonstrates a full analytics and machine learning workflow:
- framing a business problem
- inspecting and validating data
- exploring behavior patterns
- running statistical inference
- building an interpretable baseline
- comparing stronger machine learning models
- translating technical results into practical recommendations

The project also shows an important real-world lesson: **not every business problem can be solved with a highly accurate model using the data currently available**. Honest evaluation and clear communication are as important as model building.